In [7]:
import pandas as pd

df = pd.read_json(
    "/content/drive/MyDrive/Colab Notebooks/Diebetes Prediction model/train.jsonl",
    lines=True
)

print(df)

      ID  Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin  \
0    235            4      171             72             29      155   
1    686            3      130             64             29      155   
2    568            4      154             72             29      126   
3    236            7      181             84             21      192   
4    211            0      147             85             54      155   
..   ...          ...      ...            ...            ...      ...   
609  207            5      162            104             29      155   
610  388            5      144             82             26      285   
611  759            6      190             92             29      155   
612  763           10      101             76             48      180   
613   81            2       74             72             29      155   

           BMI  DiabetesPedigreeFunction  Age  Outcome  
0    43.600000                     0.479   26        1  
1    23.1

In [12]:
df.head()

,ID,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,235,4,171,72,29,155,43.6,0.479,26,1
1,686,3,130,64,29,155,23.1,0.314,22,0
2,568,4,154,72,29,126,31.3,0.338,37,0
3,236,7,181,84,21,192,35.9,0.586,51,1
4,211,0,147,85,54,155,42.8,0.375,24,0


In [13]:
df.tail()

,ID,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
609,207,5,162,104,29,155,37.700000,0.151,52,1
610,388,5,144,82,26,285,32.000000,0.452,58,1
611,759,6,190,92,29,155,35.500000,0.278,66,1
612,763,10,101,76,48,180,32.900000,0.171,63,0
613,81,2,74,72,29,155,32.457464,0.102,22,0


In [9]:
df.isnull().sum()

,0
ID,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 10 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   ID                        614 non-null    int64  
 1   Pregnancies               614 non-null    int64  
 2   Glucose                   614 non-null    int64  
 3   BloodPressure             614 non-null    int64  
 4   SkinThickness             614 non-null    int64  
 5   Insulin                   614 non-null    int64  
 6   BMI                       614 non-null    float64
 7   DiabetesPedigreeFunction  614 non-null    float64
 8   Age                       614 non-null    int64  
 9   Outcome                   614 non-null    int64  
dtypes: float64(2), int64(8)
memory usage: 48.1 KB


In [8]:
corr_matrix = df.corr()

print(corr_matrix)

                                ID  Pregnancies   Glucose  BloodPressure  \
ID                        1.000000    -0.017913  0.017698       0.035786   
Pregnancies              -0.017913     1.000000  0.134194       0.221777   
Glucose                   0.017698     0.134194  1.000000       0.223146   
BloodPressure             0.035786     0.221777  0.223146       1.000000   
SkinThickness             0.011303     0.101818  0.178557       0.206051   
Insulin                  -0.012683     0.043318  0.428063       0.070815   
BMI                       0.001196     0.034473  0.242551       0.261491   
DiabetesPedigreeFunction -0.053490    -0.041089  0.141889       0.005263   
Age                       0.019413     0.540280  0.282535       0.326436   
Outcome                  -0.050906     0.253511  0.481253       0.164499   

                          SkinThickness   Insulin       BMI  \
ID                             0.011303 -0.012683  0.001196   
Pregnancies                    0.1018

In [11]:
corr_target = df.corr()['Outcome'].sort_values(ascending=False)

print(corr_target)

Outcome                     1.000000
Glucose                     0.481253
BMI                         0.325051
Age                         0.278235
Pregnancies                 0.253511
Insulin                     0.223253
SkinThickness               0.213671
BloodPressure               0.164499
DiabetesPedigreeFunction    0.163828
ID                         -0.050906
Name: Outcome, dtype: float64


In [14]:
from sklearn.preprocessing import MinMaxScaler

In [15]:
scaler = MinMaxScaler()

# Separate features and target
X = df.drop(['ID', 'Outcome'], axis=1)
y = df['Outcome']

# Scale features to 0–1
X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

# Combine back with target if needed
df_scaled = pd.concat([X_scaled, y], axis=1)

print(df_scaled.head())

   Pregnancies   Glucose  BloodPressure  SkinThickness   Insulin       BMI  \
0     0.235294  0.819355       0.456522       0.392857  0.169471  0.616505   
1     0.176471  0.554839       0.369565       0.392857  0.169471  0.118932   
2     0.235294  0.709677       0.456522       0.392857  0.134615  0.317961   
3     0.411765  0.883871       0.586957       0.250000  0.213942  0.429612   
4     0.000000  0.664516       0.597826       0.839286  0.169471  0.597087   

   DiabetesPedigreeFunction       Age  Outcome  
0                  0.168737  0.098039        1  
1                  0.098073  0.019608        0  
2                  0.108351  0.313725        0  
3                  0.214561  0.588235        1  
4                  0.124197  0.058824        0  


In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Features and target
X = df_scaled.drop('Outcome', axis=1)
y = df_scaled['Outcome']

# Create model
model = LogisticRegression(max_iter=2000)

# Train model
model.fit(X, y)

# Predictions on training data
pred = model.predict(X)

# Accuracy
print("Accuracy:", accuracy_score(y, pred))

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y, pred))

# Detailed metrics
print("\nClassification Report:")
print(classification_report(y, pred))

Accuracy: 0.7801302931596091

Confusion Matrix:
[[369  42]
 [ 93 110]]

Classification Report:
              precision    recall  f1-score   support

           0       0.80      0.90      0.85       411
           1       0.72      0.54      0.62       203

    accuracy                           0.78       614
   macro avg       0.76      0.72      0.73       614
weighted avg       0.77      0.78      0.77       614



In [17]:
df = df.drop('ID', axis=1)

In [18]:
cols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']

for col in cols:
    df[col] = df[col].replace(0, df[col].median())

In [19]:
df["BMI_Age"] = df["BMI"] * df["Age"]

In [24]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load dataset
df = pd.read_json("/content/drive/MyDrive/Colab Notebooks/Diebetes Prediction model/train.jsonl", lines=True)

# Remove ID column
df = df.drop("ID", axis=1)

# Replace invalid zero values with median
cols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
for col in cols:
    df[col] = df[col].replace(0, df[col].median())

# Feature Engineering
df["BMI_Age"] = df["BMI"] * df["Age"]
df["Glucose_Risk"] = (df["Glucose"] > 125).astype(int)

# Separate features and target
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

# Scale features (0–1)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train Logistic Regression
model = LogisticRegression(max_iter=2000, C=0.5)
model.fit(X_scaled, y)

# Predictions
pred = model.predict(X_scaled)

# Evaluation
print("Accuracy:", accuracy_score(y, pred))
print("\nConfusion Matrix:\n", confusion_matrix(y, pred))
print("\nClassification Report:\n", classification_report(y, pred))

# Feature Importance
importance = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

print("\nFeature Importance:")
print(importance.sort_values(by="Coefficient", ascending=False))

Accuracy: 0.7671009771986971

Confusion Matrix:
 [[363  48]
 [ 95 108]]

Classification Report:
               precision    recall  f1-score   support

           0       0.79      0.88      0.84       411
           1       0.69      0.53      0.60       203

    accuracy                           0.77       614
   macro avg       0.74      0.71      0.72       614
weighted avg       0.76      0.77      0.76       614


Feature Importance:
                    Feature  Coefficient
1                   Glucose     2.315698
5                       BMI     1.719376
8                   BMI_Age     1.361958
0               Pregnancies     1.256401
6  DiabetesPedigreeFunction     0.856013
9              Glucose_Risk     0.651969
3             SkinThickness     0.474050
7                       Age     0.428239
4                   Insulin     0.330250
2             BloodPressure    -0.191204


In [36]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Load dataset
df = pd.read_json("/content/drive/MyDrive/Colab Notebooks/Diebetes Prediction model/train.jsonl", lines=True)

# Remove ID column
df = df.drop("ID", axis=1)

# Replace invalid zero values
cols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']
for col in cols:
    df[col] = df[col].replace(0, df[col].median())

# Select important features
X = df[['Glucose','BMI','Age','Pregnancies','Insulin']]
y = df['Outcome']

# Scaling (not required for Random Forest but safe)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Random Forest model
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=40
)

# Train
model.fit(X_scaled, y)

# Prediction
pred = model.predict(X_scaled)

# Evaluation
print("Accuracy:", accuracy_score(y, pred))
print("\nConfusion Matrix:\n", confusion_matrix(y, pred))
print("\nClassification Report:\n", classification_report(y, pred))

Accuracy: 0.8843648208469055

Confusion Matrix:
 [[375  36]
 [ 35 168]]

Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.91      0.91       411
           1       0.82      0.83      0.83       203

    accuracy                           0.88       614
   macro avg       0.87      0.87      0.87       614
weighted avg       0.88      0.88      0.88       614



In [39]:
import joblib

# Save model
joblib.dump(model, "diabetes_model.pkl")
joblib.dump(scaler, "scaler.pkl")


['scaler.pkl']

In [37]:
test = pd.read_json("/content/test.jsonl", lines=True)

# Remove ID
test = test.drop("ID", axis=1)

# Same preprocessing
for col in cols:
    test[col] = test[col].replace(0, test[col].median())

X_test = test[['Glucose','BMI','Age','Pregnancies','Insulin']]
y_test = test['Outcome']

# Scale test data
X_test_scaled = scaler.transform(X_test)

# Prediction
pred = model.predict(X_test_scaled)

# Accuracy
print("Test Accuracy:", accuracy_score(y_test, pred))

Test Accuracy: 0.7922077922077922
